# 01 - Explorative Datenanalyse

Analyse der verfügbaren Datenquellen für die PV-Optimierung:
- **Wetter** (Open-Meteo): Sonneneinstrahlung, Temperatur, Bewölkung
- **Strompreise** (aWATTar): Day-Ahead-Preise, negative Preise
- HA/evcc Daten: *sobald verfügbar ergänzen*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

## 1. Historische Wetterdaten laden

In [ ]:
weather = pd.read_csv("../data/raw/weather_historical_30d.csv", parse_dates=["timestamp"])
print(f"Zeitraum: {weather.timestamp.min()} bis {weather.timestamp.max()}")
print(f"Zeilen: {len(weather)}")
weather.describe()

## 2. Sonneneinstrahlung über 30 Tage

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(weather.timestamp, weather.shortwave_radiation, color="orange", alpha=0.7)
axes[0].set_ylabel("GHI (W/m²)")
axes[0].set_title("Globalstrahlung (Shortwave Radiation)")
axes[0].fill_between(weather.timestamp, weather.shortwave_radiation, alpha=0.3, color="orange")

axes[1].plot(weather.timestamp, weather.direct_radiation, color="red", alpha=0.7)
axes[1].set_ylabel("DNI (W/m²)")
axes[1].set_title("Direktstrahlung")
axes[1].fill_between(weather.timestamp, weather.direct_radiation, alpha=0.3, color="red")

axes[2].plot(weather.timestamp, weather.cloud_cover, color="gray", alpha=0.7)
axes[2].set_ylabel("Bewölkung (%)")
axes[2].set_title("Bewölkungsgrad")
axes[2].fill_between(weather.timestamp, weather.cloud_cover, alpha=0.3, color="gray")

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))

plt.tight_layout()
plt.show()

## 3. Tagesprofile: Strahlung nach Tageszeit

In [ ]:
weather["hour"] = weather.timestamp.dt.hour

hourly_avg = weather.groupby("hour")[["shortwave_radiation", "direct_radiation", "diffuse_radiation"]].mean()

fig, ax = plt.subplots(figsize=(10, 5))
hourly_avg.plot(ax=ax, marker="o")
ax.set_xlabel("Stunde")
ax.set_ylabel("Strahlung (W/m²)")
ax.set_title("Durchschnittliche Sonneneinstrahlung nach Tageszeit (30 Tage)")
ax.set_xticks(range(0, 24))
ax.legend(["GHI (Global)", "Direkt", "Diffus"])
plt.tight_layout()
plt.show()

## 4. Korrelation: Strahlung vs. Temperatur vs. Bewölkung

In [ ]:
# Nur Tagesstunden (6-20 Uhr)
day = weather[(weather.hour >= 6) & (weather.hour <= 20)].copy()

corr_cols = ["shortwave_radiation", "direct_radiation", "diffuse_radiation", 
             "cloud_cover", "temperature_2m", "wind_speed_10m"]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(day[corr_cols].corr(), annot=True, fmt=".2f", cmap="RdYlBu_r", ax=ax, vmin=-1, vmax=1)
ax.set_title("Korrelation der Wettervariablen (Tagesstunden)")
plt.tight_layout()
plt.show()

## 5. Tägliche Strahlungsenergie (potentieller PV-Ertrag)

In [ ]:
weather["date"] = weather.timestamp.dt.date
daily = weather.groupby("date").agg(
    total_ghi_wh=pd.NamedAgg(column="shortwave_radiation", aggfunc="sum"),  # W/m² * 1h ≈ Wh/m²
    max_ghi=pd.NamedAgg(column="shortwave_radiation", aggfunc="max"),
    mean_temp=pd.NamedAgg(column="temperature_2m", aggfunc="mean"),
    mean_cloud=pd.NamedAgg(column="cloud_cover", aggfunc="mean"),
).reset_index()

daily["total_ghi_kwh"] = daily.total_ghi_wh / 1000  # kWh/m² pro Tag

fig, ax1 = plt.subplots(figsize=(14, 5))
bars = ax1.bar(daily.date, daily.total_ghi_kwh, color="orange", alpha=0.7, label="Tägliche GHI (kWh/m²)")
ax1.set_ylabel("Strahlungsenergie (kWh/m²)")
ax1.set_title("Tägliche Globalstrahlung — Potentieller PV-Ertrag")

ax2 = ax1.twinx()
ax2.plot(daily.date, daily.mean_cloud, color="gray", marker="o", markersize=3, label="Bewölkung (%)")
ax2.set_ylabel("Bewölkung (%)")

fig.legend(loc="upper right", bbox_to_anchor=(0.95, 0.95))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Durchschnittliche tägliche Globalstrahlung: {daily.total_ghi_kwh.mean():.1f} kWh/m²")
print(f"Bester Tag: {daily.loc[daily.total_ghi_kwh.idxmax(), 'date']} mit {daily.total_ghi_kwh.max():.1f} kWh/m²")
print(f"Schlechtester Tag: {daily.loc[daily.total_ghi_kwh.idxmin(), 'date']} mit {daily.total_ghi_kwh.min():.1f} kWh/m²")

## 6. Strompreise

In [ ]:
prices = pd.read_csv("../data/raw/prices_awattar.csv", parse_dates=["timestamp"])

fig, ax = plt.subplots(figsize=(12, 4))

colors = ["red" if neg else "steelblue" for neg in prices.is_negative]
ax.bar(prices.timestamp, prices.price_eur_mwh, color=colors, width=0.03)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_ylabel("Preis (EUR/MWh)")
ax.set_title("Day-Ahead Strompreise (aWATTar)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

plt.tight_layout()
plt.show()

print(f"Aktueller Preisbereich: {prices.price_eur_mwh.min():.1f} – {prices.price_eur_mwh.max():.1f} EUR/MWh")
print(f"Negative Preise: {prices.is_negative.sum()} Stunden")

## 7. Zusammenfassung & nächste Schritte

### Erkenntnisse
- Sonneneinstrahlung (GHI) ist der wichtigste Prädiktor für PV-Ertrag
- Starke negative Korrelation zwischen Bewölkung und Strahlung
- Klare Tagesprofile: Peak zwischen 11-14 Uhr
- Erhebliche Tagesunterschiede in der Strahlungsenergie

### Nächste Schritte
1. HA/evcc Daten ergänzen (echte PV-Erzeugung, Batterie, Verbrauch)
2. Feature Engineering für ML-Modell
3. Erstes LightGBM-Modell trainieren (mit Wetter→PV-Korrelation)